# Data Preprocessing

In this notebook, I go through the dataset and try to understand it first. If needed, I preprocess the messy data so that later the agent can use it effectively without issues. This also helps speed up the development process.

Date: 2026-10-22

## 1. Import Libraries & Load The Dataset

In [1]:
from shutil import which

import pandas as pd

print("Pandas version:", pd.__version__)

Pandas version: 3.0.6


In [2]:
shops = pd.read_csv("raw/shops.csv", dtype=str)
offers = pd.read_csv("raw/offers.csv", dtype=str)
products = pd.read_csv("raw/products.csv", dtype=str)

print("shops shape:", shops.shape)
print("offers shape:", offers.shape)
print("products shape:", products.shape)

shops shape: (6476, 13)
offers shape: (331506, 8)
products shape: (225903, 7)


## 2. Data Understanding

First I want to see some examples of the data.

In [3]:
shops.head(2)

,address,category_names,contacts,instagram,lat,location_label,lon,name,payment_options,shop_id,supports_fast_delivery,website_url,weekly_hours
0,خیابان جمهوری، نبش شیخ هادی، پاساژ علالدین 2، ...,"[""دوربین مدار بسته"",""ضبط کننده ویدیویی دوربین‌...","[{""channel"":""phone"",""display_number"":""۰۲۱۲۶۴۶۷...",NaN,35.694991526310005,شیخ هادی,51.407137248405895,فرورا رویس,[],shop_3f93a30a9e66c6dece07cdff,NaN,NaN,"{""fri"":{""shifts"":[{""end_time"":""00:00:00"",""star..."
1,خیابان جمهوری، نرسیده به پل حافظ، نبش کوچه کلا...,"[""گوشی موبایل"",""لوازم جانبی موبایل و تبلت""]","[{""channel"":""phone"",""display_number"":""۰۲۱۷۶۵۵۷...",NaN,35.695329620242966,شیخ هادی,51.410700800954594,موبایل تخت جمشید,[],shop_51d0c1a6edf81ba759726bab,NaN,https://takhtjamshidmobile.com/,{}


Interesting! There are things to handle. We need to investigate each column individually.

I need to normalize Persian texts, most probably we have inconsistency in addresses and categories. contacts are list of JSON! Payments is a list, shifts are JOSN. We have a long way to go with this CSV...

In [4]:
offers.head(2)

,description,last_price_changed_at,offer_id,price_toman,product_id,seller_title,shop_id,warranty
0,NaN,2026-09-14T08:57:27.401253+00:00,offer_98512b2127469a2d20a64b3b,610000,product_af3172caa21dd9a3e40e6349,کوپلر برنجی - اتصال دهنده میله ارت - آچارخور س...,shop_d9a17ad60b0fd5c15bbe2f0a,NaN
1,NaN,2026-09-14T08:58:36.014395+00:00,offer_d78b4c5666c03670580249f2,200000000,product_052451f267a741bd28c12cef,صاعقه گیر الکترونیکی - برند شرتک - ساخت اتریش 15,shop_d9a17ad60b0fd5c15bbe2f0a,NaN


Nice! We have last_price_change which is super important for this product. I need to investigate it more. Other columns need to be checked as well. Let's see what we have :)))

In [5]:
products.head(2)

,brand_name,category_name,image_urls,model,persian_title,product_id,specs
0,سونی,لنز دوربین,"[""https://image.torob.com/base/images/Ih/rU/Ih...",NaN,لنز دوربین سونی مدل E 70-350mm f/4.5-6.3 G OSS,product_47796232454501a5e8179e26,"[{""key"":""نام محصول"",""section"":""مشخصات کلی"",""va..."
1,NaN,سیم و کابل برق ساختمان,"[""https://image.torob.com/base/images/8G/Da/8G...",NaN,کابل افشان ۱ متری ۱۶×۵ مشهد با روکش PVC,product_ca6c23c9f3f4dbcf72d3894d,"[{""key"":""برند"",""section"":""مشخصات کلی"",""value"":..."


Again lots of Persian texts and list of JSONs. All needs to be investigated further.

## 3. Understand Missing Values

Since we have list and JSON in the dataset, we need to tell pandas to consider them as NA is they are empty so that we can analyze the missing values.

In [6]:
empty_values = ["[]", "{}"]

shops = shops.replace(empty_values, pd.NA)
offers = offers.replace(empty_values, pd.NA)
products = products.replace(empty_values, pd.NA)

In [7]:
shops.isna().mean()

address                   0.000154
category_names            0.277486
contacts                  0.000000
instagram                 0.996603
lat                       0.000000
location_label            0.000154
lon                       0.000000
name                      0.000000
payment_options           0.958925
shop_id                   0.000000
supports_fast_delivery    0.752779
website_url               0.986103
weekly_hours              0.309296
dtype: float64

In [8]:
offers.isna().mean()

description              0.884234
last_price_changed_at    0.002320
offer_id                 0.000000
price_toman              0.045498
product_id               0.000000
seller_title             0.000000
shop_id                  0.000000
warranty                 0.822196
dtype: float64

In [9]:
products.isna().mean()

brand_name       0.744873
category_name    0.000000
image_urls       0.005396
model            0.982045
persian_title    0.000009
product_id       0.000000
specs            0.359907
dtype: float64

## 4. Handle Data Types
Since I load CSV files as strings, I want to convert them to appropriate data types.

In [10]:
# Longitude and latitude as decimal so later we can compute how far a shop is from a customer
shops["lat"] = shops["lat"].astype(float)
shops["lon"] = shops["lon"].astype(float)

# Covert this column to True, False, or None so later life would be easier :D
shops["supports_fast_delivery"] = (
    shops["supports_fast_delivery"]
    .map({"true": True, "false": False})
    .astype("boolean")
)

offers["price_toman"] = offers["price_toman"].astype("Int64")

# I kept time zone UTC, so everything is consistent, later I easily convert it when needed
offers["last_price_changed_at"] = pd.to_datetime(
    offers["last_price_changed_at"], utc=True
)

In [11]:
shops.dtypes

address                       str
category_names                str
contacts                      str
instagram                     str
lat                       float64
location_label                str
lon                       float64
name                          str
payment_options               str
shop_id                       str
supports_fast_delivery    boolean
website_url                   str
weekly_hours                  str
dtype: object

In [12]:
offers.dtypes

description                              str
last_price_changed_at    datetime64[us, UTC]
offer_id                                 str
price_toman                            Int64
product_id                               str
seller_title                             str
shop_id                                  str
warranty                                 str
dtype: object

## 5. Persian Text Normalization

My favorite part when working with Persian text is normalization. We don't use RAG here, so maybe this is not necessary, but I am obsessed with having cleaned data :)
So let's handle arabic characters, numbers, and missing invisible characters which I really hate.

In [13]:
import re

In [14]:
# Map one character to appropriate target
DIGITS_AND_LETTERS = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹" + "٠١٢٣٤٥٦٧٨٩" + "يىك" + "\u200c\u00a0",
    "0123456789" + "0123456789" + "ییک" + "  ",
)

# Characters to delete
INVISIBLE = re.compile(r"[\u064b-\u0652\u0640\u200b\u200e\u200f\ufeff]")

def normalize_fa(text):
    if pd.isna(text):
        return text
    text = text.translate(DIGITS_AND_LETTERS)
    text = INVISIBLE.sub("", text)
    text = " ".join(text.split())
    return text

In [15]:
# Test the function
normalize_fa("سبك فرمان  راست - تیگو ۷\u200cاکسلنت\n")

'سبک فرمان راست - تیگو 7 اکسلنت'

In [16]:
# Apply the regex
for col in ["name", "address", "location_label"]:
    shops[col] = shops[col].map(normalize_fa)

for col in ["persian_title", "brand_name", "category_name", "model"]:
    products[col] = products[col].map(normalize_fa)

offers["seller_title"] = offers["seller_title"].map(normalize_fa)

In [17]:
# Check the results
pattern = "ي|ك|[۰-۹]"
print(products["persian_title"].str.contains(pattern).sum())
print(offers["seller_title"].str.contains(pattern).sum())
print(shops["address"].str.contains(pattern).sum())

0
0
0


## 6. Create a New Table for Working Hours

First I start with unpacking `weekly_hours` in shops CSV. Let's see what we have there.

In [18]:
import json

In [19]:
rows = []
for shop_id, hours_json in zip(shops["shop_id"], shops["weekly_hours"]):
    if pd.isna(hours_json):
        continue
    week = json.loads(hours_json)
    for day, info in week.items():
        for shift in info["shifts"]:
            rows.append({
                "shop_id": shop_id,
                "day": day,
                "status": info["status"],
                "start_time": shift["start_time"],
                "end_time": shift["end_time"],
            })

shop_working_hours = pd.DataFrame(rows)
print("Total shop working hour records:", f"{len(shop_working_hours):,}")

Total shop working hour records: 58,786


In [20]:
shop_working_hours.head(2)

,shop_id,day,status,start_time,end_time
0,shop_3f93a30a9e66c6dece07cdff,fri,closed,00:00:00,00:00:00
1,shop_3f93a30a9e66c6dece07cdff,mon,declared,10:00:00,19:00:00


In [21]:
# Investigating on status values
shop_working_hours["status"].value_counts()

status
declared       51066
closed          7664
unsupported       56
Name: count, dtype: int64

In [23]:
# Investigate on start times
shop_working_hours["start_time"].value_counts().head()

start_time
00:00:00    31493
09:00:00    10604
10:00:00     6998
09:30:00     3519
10:30:00     1788
Name: count, dtype: int64

start_time
00:00:00    31493
09:00:00    10604
10:00:00     6998
09:30:00     3519
10:30:00     1788
Name: count, dtype: int64

DAMN! what is that?! 31.5K shops open at midnight? something is wtrong here...

In [25]:
# A random shop's data
example = shops["weekly_hours"].dropna().sample(1).iloc[0]

week = json.loads(example)

print(week["sat"])
print(week["fri"])

{'shifts': [{'end_time': '23:00:00', 'start_time': '20:00:00'}, {'end_time': '00:00:00', 'start_time': '00:00:00'}], 'status': 'declared'}
{'shifts': [{'end_time': '00:00:00', 'start_time': '00:00:00'}, {'end_time': '00:00:00', 'start_time': '00:00:00'}], 'status': 'closed'}
{'shifts': [{'end_time': '19:00:00', 'start_time': '09:00:00'}, {'end_time': '00:00:00', 'start_time': '00:00:00'}], 'status': 'declared'}
{'shifts': [{'end_time': '00:00:00', 'start_time': '00:00:00'}, {'end_time': '00:00:00', 'start_time': '00:00:00'}], 'status': 'closed'}


We are facing 2 values for each shift data. I don't understand why. I can clearly sat 00:00 start and end time is a placeholder for closed days such as Friday.

In [27]:
# Flag for placeholder shifts
is_placeholder = (shop_working_hours["start_time"] == "00:00:00") & (shop_working_hours["end_time"] == "00:00:00")

# Find shops where 2nd shift is not 00:00
shift_number = shop_working_hours.groupby(["shop_id", "day"]).cumcount() + 1
has_second_shift = shop_working_hours[(shift_number == 2) & ~is_placeholder]


has_second_shift.head()

,shop_id,day,status,start_time,end_time
500,shop_cc3afa4bbae2b5dc09dcf1d5,mon,declared,18:00:00,21:00:00
502,shop_cc3afa4bbae2b5dc09dcf1d5,sat,declared,18:00:00,21:00:00
504,shop_cc3afa4bbae2b5dc09dcf1d5,sun,declared,18:00:00,21:00:00
506,shop_cc3afa4bbae2b5dc09dcf1d5,thu,declared,18:00:00,21:00:00
508,shop_cc3afa4bbae2b5dc09dcf1d5,tue,declared,18:00:00,21:00:00


,shop_id,day,status,start_time,end_time
500,shop_cc3afa4bbae2b5dc09dcf1d5,mon,declared,18:00:00,21:00:00
502,shop_cc3afa4bbae2b5dc09dcf1d5,sat,declared,18:00:00,21:00:00
504,shop_cc3afa4bbae2b5dc09dcf1d5,sun,declared,18:00:00,21:00:00
506,shop_cc3afa4bbae2b5dc09dcf1d5,thu,declared,18:00:00,21:00:00
508,shop_cc3afa4bbae2b5dc09dcf1d5,tue,declared,18:00:00,21:00:00


In [29]:
# Print a single day shift for above example
example = has_second_shift.iloc[0]
shop_working_hours[(shop_working_hours["shop_id"] == example["shop_id"]) & (shop_working_hours["day"] == example["day"])]

,shop_id,day,status,start_time,end_time
499,shop_cc3afa4bbae2b5dc09dcf1d5,mon,declared,10:00:00,16:00:00
500,shop_cc3afa4bbae2b5dc09dcf1d5,mon,declared,18:00:00,21:00:00


,shop_id,day,status,start_time,end_time
499,shop_cc3afa4bbae2b5dc09dcf1d5,mon,declared,10:00:00,16:00:00
500,shop_cc3afa4bbae2b5dc09dcf1d5,mon,declared,18:00:00,21:00:00


Lovely! This example shows they have a shift from 10:00 to 16:00, one fom 18:00 to 21:00

It is a lunch break or rest between shifts. That is great! Now I clearly understand why we have 2 shift data per day!

In [31]:
# Let's see this pattern across all shops
pd.crosstab(shop_working_hours["status"], is_placeholder)

col_0,False,True
status,,
closed,0,7664
declared,27737,23329
unsupported,39,17


col_0,False,True
status,,
closed,0,7664
declared,27737,23329
unsupported,39,17


Every closed day has only placeholder which makes sense.

23, 329 open days also have a placeholder. That's alot. I prefer to fix this because this is junk!

In [33]:
declared = shop_working_hours[shop_working_hours["status"] == "declared"]
only_placeholder = is_placeholder[declared.index].groupby([declared["shop_id"], declared["day"]]).all()
print(only_placeholder.sum())

0
0


Deleting is safe. Every open day still has at least one real shift. I'll do it.

In [35]:
# How unsupported look like?
shop_working_hours[shop_working_hours["status"] == "unsupported"].head()

,shop_id,day,status,start_time,end_time
8,shop_9900a46f54a879ce260839bb,mon,unsupported,08:00:00,08:00:00
9,shop_9900a46f54a879ce260839bb,sat,unsupported,08:00:00,08:00:00
10,shop_9900a46f54a879ce260839bb,sun,unsupported,08:00:00,08:00:00
12,shop_9900a46f54a879ce260839bb,tue,unsupported,08:00:00,08:00:00
13,shop_9900a46f54a879ce260839bb,wed,unsupported,08:00:00,08:00:00


,shop_id,day,status,start_time,end_time
8,shop_9900a46f54a879ce260839bb,mon,unsupported,08:00:00,08:00:00
9,shop_9900a46f54a879ce260839bb,sat,unsupported,08:00:00,08:00:00
10,shop_9900a46f54a879ce260839bb,sun,unsupported,08:00:00,08:00:00
12,shop_9900a46f54a879ce260839bb,tue,unsupported,08:00:00,08:00:00
13,shop_9900a46f54a879ce260839bb,wed,unsupported,08:00:00,08:00:00


These are also meaningless. I prefer to keep status, but treat time as unknown :)

In [37]:
# This is the final check. Do we have shops open at midnight?
real = shop_working_hours[(shop_working_hours["status"] == "declared") & ~is_placeholder]
print((real["end_time"] <= real["start_time"]).sum())

0
0


In [39]:
# Data transformation

# Remove placeholder shifts on open days
shop_working_hours = shop_working_hours[~(is_placeholder & (shop_working_hours["status"] == "declared"))].copy()

# Handle closed and unsupported days
shop_working_hours.loc[shop_working_hours["status"] != "declared", ["start_time", "end_time"]] = pd.NA

# Remove duplicate repeated shifts
shop_working_hours = shop_working_hours.drop_duplicates()

# Normalize time, convert "10:00:00" to "10:00"
shop_working_hours["start_time"] = shop_working_hours["start_time"].str[:5]
shop_working_hours["end_time"] = shop_working_hours["end_time"].str[:5]

/var/folders/by/gpy__z0527d5r9djzctznr780000gn/T/ipykernel_73572/4163080869.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  shop_working_hours = shop_working_hours[~(is_placeholder & (shop_working_hours["status"] == "declared"))].copy()


In [41]:
shop_working_hours["status"].value_counts()

status
declared       27689
closed          4006
unsupported       35
Name: count, dtype: int64

status
declared       27689
closed          4006
unsupported       35
Name: count, dtype: int64

In [43]:
shop_working_hours["start_time"].value_counts().head()

start_time
09:00    10575
10:00     6987
09:30     3502
10:30     1788
11:00     1763
Name: count, dtype: int64

start_time
09:00    10575
10:00     6987
09:30     3502
10:30     1788
11:00     1763
Name: count, dtype: int64

Yes! this is correct! Now I feel much better :)

In [45]:
# Let's check random shop from above
shop_working_hours[shop_working_hours["shop_id"] == "shop_cc3afa4bbae2b5dc09dcf1d5"]

,shop_id,day,status,start_time,end_time
497,shop_cc3afa4bbae2b5dc09dcf1d5,fri,closed,NaN,NaN
499,shop_cc3afa4bbae2b5dc09dcf1d5,mon,declared,10:00,16:00
500,shop_cc3afa4bbae2b5dc09dcf1d5,mon,declared,18:00,21:00
501,shop_cc3afa4bbae2b5dc09dcf1d5,sat,declared,10:00,16:00
502,shop_cc3afa4bbae2b5dc09dcf1d5,sat,declared,18:00,21:00
503,shop_cc3afa4bbae2b5dc09dcf1d5,sun,declared,10:00,16:00
504,shop_cc3afa4bbae2b5dc09dcf1d5,sun,declared,18:00,21:00
505,shop_cc3afa4bbae2b5dc09dcf1d5,thu,declared,10:00,16:00
506,shop_cc3afa4bbae2b5dc09dcf1d5,thu,declared,18:00,21:00
507,shop_cc3afa4bbae2b5dc09dcf1d5,tue,declared,10:00,16:00


,shop_id,day,status,start_time,end_time
497,shop_cc3afa4bbae2b5dc09dcf1d5,fri,closed,NaN,NaN
499,shop_cc3afa4bbae2b5dc09dcf1d5,mon,declared,10:00,16:00
500,shop_cc3afa4bbae2b5dc09dcf1d5,mon,declared,18:00,21:00
501,shop_cc3afa4bbae2b5dc09dcf1d5,sat,declared,10:00,16:00
502,shop_cc3afa4bbae2b5dc09dcf1d5,sat,declared,18:00,21:00
503,shop_cc3afa4bbae2b5dc09dcf1d5,sun,declared,10:00,16:00
504,shop_cc3afa4bbae2b5dc09dcf1d5,sun,declared,18:00,21:00
505,shop_cc3afa4bbae2b5dc09dcf1d5,thu,declared,10:00,16:00
506,shop_cc3afa4bbae2b5dc09dcf1d5,thu,declared,18:00,21:00
507,shop_cc3afa4bbae2b5dc09dcf1d5,tue,declared,10:00,16:00


## 7. Create a New Table for Shop Payments


In [47]:
rows = []
for shop_id, payments_json in zip(shops["shop_id"], shops["payment_options"]):
    if pd.isna(payments_json):
        continue
    for option in json.loads(payments_json):
        rows.append({
            "shop_id": shop_id,
            "provider": option["provider_name"],
            "channel": option["channel"],
            "category_names": option["category_names"],
            "min_price_toman": option["min_price_toman"],
            "max_price_toman": option["max_price_toman"],
        })

shop_payments = pd.DataFrame(rows)
print("Total shop payment records:", f"{len(shop_payments):,}")

Total shop payment records: 385
Total shop payment records: 385


A small fraction of shops has payment info in the dataset.

In [49]:
shop_payments.head()

,shop_id,provider,channel,category_names,min_price_toman,max_price_toman
0,shop_46d86ffd33a1fcf312304791,اسنپ‌پی,unknown,[],NaN,NaN
1,shop_46d86ffd33a1fcf312304791,ترب‌پی,unknown,[],NaN,NaN
2,shop_6b872abfb4fde6c2d7f2f979,ترب‌پی,unknown,[],NaN,NaN
3,shop_5d31d837b37ca66ebf9bc25c,اسنپ‌پی,unknown,[],NaN,NaN
4,shop_b27a7e52c1289e75c3810b34,دیجی پی,in_person,[],NaN,NaN


,shop_id,provider,channel,category_names,min_price_toman,max_price_toman
0,shop_46d86ffd33a1fcf312304791,اسنپ‌پی,unknown,[],NaN,NaN
1,shop_46d86ffd33a1fcf312304791,ترب‌پی,unknown,[],NaN,NaN
2,shop_6b872abfb4fde6c2d7f2f979,ترب‌پی,unknown,[],NaN,NaN
3,shop_5d31d837b37ca66ebf9bc25c,اسنپ‌پی,unknown,[],NaN,NaN
4,shop_b27a7e52c1289e75c3810b34,دیجی پی,in_person,[],NaN,NaN


In [51]:
shop_payments["provider"].value_counts()

provider
ترب‌پی                   190
اسنپ‌پی                   89
دیجی پی                   61
تارا                      37
ازکی‌وام                   5
دکان ابزار                 1
تجهیزات آشپزخانه حلما      1
جیتکس مارکت                1
Name: count, dtype: int64

provider
ترب‌پی                   190
اسنپ‌پی                   89
دیجی پی                   61
تارا                      37
ازکی‌وام                   5
دکان ابزار                 1
تجهیزات آشپزخانه حلما      1
جیتکس مارکت                1
Name: count, dtype: int64

Hmmmm, look at the last 3 providers. They don't look like legit payment provider, but I don't remove them. Maybe it's their own installment plan. Only 1 count for each so I keep them.

Also we need to normalize names for that half space :)

In [53]:
# Normalize name
shop_payments["provider"] = shop_payments["provider"].map(normalize_fa)

# Let's understand channels. This also directly effect the performance of our agent
pd.crosstab(shop_payments["provider"], shop_payments["channel"])

channel,in_person,unknown
provider,,
ازکی وام,0,5
اسنپ پی,53,36
تارا,26,11
تجهیزات آشپزخانه حلما,0,1
ترب پی,50,140
جیتکس مارکت,0,1
دکان ابزار,0,1
دیجی پی,45,16


channel,in_person,unknown
provider,,
ازکی وام,0,5
اسنپ پی,53,36
تارا,26,11
تجهیزات آشپزخانه حلما,0,1
ترب پی,50,140
جیتکس مارکت,0,1
دکان ابزار,0,1
دیجی پی,45,16


Now let's check are other columns useful?

In [55]:
print((shop_payments["category_names"].str.len() > 0).sum())
print(((shop_payments["min_price_toman"] > 0) | (shop_payments["max_price_toman"] > 0)).sum())

1
2
1
2


Out of 385 rows, only 1 has categories and only 2 have a price limit. Those columns are almost always empty, so we drop them. I remove them.

In [57]:
shop_payments = shop_payments[["shop_id", "provider", "channel"]].copy()

## 8. Create a New Table for Shop Categories

In [59]:
rows = []
for shop_id, categories_json in zip(shops["shop_id"], shops["category_names"]):
    if pd.isna(categories_json):
        continue
    for category in json.loads(categories_json):
        rows.append({"shop_id": shop_id, "category": category})

shop_categories = pd.DataFrame(rows)
print("Total shop category records:", f"{len(shop_categories):,}")


Total shop category records: 64,659
Total shop category records: 64,659


In [61]:
shop_categories.head()

,shop_id,category
0,shop_3f93a30a9e66c6dece07cdff,دوربین مدار بسته
1,shop_3f93a30a9e66c6dece07cdff,ضبط کننده ویدیویی دوربین‌های نظارتی و امنیتی
2,shop_3f93a30a9e66c6dece07cdff,دوربین خودرو و موتور سیکلت
3,shop_3f93a30a9e66c6dece07cdff,دوربین و مانیتور کنترل کودک
4,shop_3f93a30a9e66c6dece07cdff,دوربین امنیتی خورشیدی


,shop_id,category
0,shop_3f93a30a9e66c6dece07cdff,دوربین مدار بسته
1,shop_3f93a30a9e66c6dece07cdff,ضبط کننده ویدیویی دوربین‌های نظارتی و امنیتی
2,shop_3f93a30a9e66c6dece07cdff,دوربین خودرو و موتور سیکلت
3,shop_3f93a30a9e66c6dece07cdff,دوربین و مانیتور کنترل کودک
4,shop_3f93a30a9e66c6dece07cdff,دوربین امنیتی خورشیدی


In [63]:
# Normalize category names as before
shop_categories["category"] = shop_categories["category"].map(normalize_fa)

In [65]:
# How many categories each shop has
shop_categories.groupby("shop_id").size().describe()

count    4679.000000
mean       13.818978
std        24.983884
min         1.000000
25%         2.000000
50%         5.000000
75%        14.000000
max       363.000000
dtype: float64

count    4679.000000
mean       13.818978
std        24.983884
min         1.000000
25%         2.000000
50%         5.000000
75%        14.000000
max       363.000000
dtype: float64

WOW! one shop has 363 categories :D That is weird...

In [67]:
# Do shop categories use the same names as product categories?
product_categories = set(products["category_name"])
print(f"{shop_categories['category'].isin(product_categories).mean():.2f}")

0.96
0.96


Good! 96% of shop categories use same name as product categories. Finally something makes sense.

In [69]:
# Now let's see if those shops have real products to sell
has_categories = shops["category_names"].notna()
has_offers = shops["shop_id"].isin(offers["shop_id"])

pd.crosstab(has_categories, has_offers)

shop_id,False,True
category_names,,
False,1797,0
True,887,3792


shop_id,False,True
category_names,,
False,1797,0
True,887,3792


- 3,792 have real products with prices.
- 887 only say what they sell. They have no products in the data.
- 1,797 have nothing but a location (and maybe hours).

Let's add this to the table as a new column, it is useful for sure!

In [71]:
shops["shop_type"] = "location_only"
shops.loc[has_categories, "shop_type"] = "category_only"
shops.loc[has_offers, "shop_type"] = "has_products"

shops["shop_type"].value_counts()

shop_type
has_products     3792
location_only    1797
category_only     887
Name: count, dtype: int64

shop_type
has_products     3792
location_only    1797
category_only     887
Name: count, dtype: int64

## 9. Create a New Table for Warranty

In [73]:
warranty = offers["warranty"].dropna().map(json.loads)
warranty = pd.DataFrame(warranty.tolist(), index=warranty.index)

print("Total warranty records:", f"{len(warranty):,}")

Total warranty records: 58,943
Total warranty records: 58,943


In [75]:
warranty.head()

,description,period_months,provider
17,NaN,NaN,تست و تحویل در حضور مشتری
23,NaN,6.0,شرکتی
53,NaN,NaN,بدون گارانتی
54,NaN,NaN,بدون گارانتی
56,NaN,NaN,بدون گارانتی


,description,period_months,provider
17,NaN,NaN,تست و تحویل در حضور مشتری
23,NaN,6.0,شرکتی
53,NaN,NaN,بدون گارانتی
54,NaN,NaN,بدون گارانتی
56,NaN,NaN,بدون گارانتی


In [77]:
# Let's understand the records more
warranty["provider"].value_counts().head(10)

provider
شرکتی                            7310
تست و تحویل در حضور مشتری        3977
بدون گارانتی                     1625
پارس ارتباط افزار                 292
انتخاب سرویس حامی                 198
مادیران                           122
آونگ                               86
ژیوار فراگستر داریا الکترونیک      83
سام سرویس                          72
مشتریان گلدیران                    69
Name: count, dtype: int64

provider
شرکتی                            7310
تست و تحویل در حضور مشتری        3977
بدون گارانتی                     1625
پارس ارتباط افزار                 292
انتخاب سرویس حامی                 198
مادیران                           122
آونگ                               86
ژیوار فراگستر داریا الکترونیک      83
سام سرویس                          72
مشتریان گلدیران                    69
Name: count, dtype: int64

In [79]:
warranty["description"].value_counts().head(15)

description
گارانتی اصالت و سلامت فیزیکی کالا     2058
ضمانت اصالت کالا                      1407
ضمانت تعویض کالا تا 7 روز              857
اصالت و سلامت فیزیکی کالا              839
2 سال                                  784
ضمانت اصالت و سلامت فیزیکی کالا        723
تضمین اصالت کالا                       622
اصالت و سلامت فیزیکی                   549
5 سال                                  477
ضمانت اصالت و سلامت کالا               444
ضمانت بازگشت کالا                      436
گارانتی اصالت و سلامتی فیزیکی کالا     414
12 ماه                                 390
18 ماهه دیباکام پردازش سدرا            386
None                                   373
Name: count, dtype: int64

description
گارانتی اصالت و سلامت فیزیکی کالا     2058
ضمانت اصالت کالا                      1407
ضمانت تعویض کالا تا 7 روز              857
اصالت و سلامت فیزیکی کالا              839
2 سال                                  784
ضمانت اصالت و سلامت فیزیکی کالا        723
تضمین اصالت کالا                       622
اصالت و سلامت فیزیکی                   549
5 سال                                  477
ضمانت اصالت و سلامت کالا               444
ضمانت بازگشت کالا                      436
گارانتی اصالت و سلامتی فیزیکی کالا     414
12 ماه                                 390
18 ماهه دیباکام پردازش سدرا            386
None                                   373
Name: count, dtype: int64

Let's turn the warranty into 3 columns on `offers`: provider, months, description, turn the text "None" into a real missing value, and ofc normalize the Persian text :D

In [81]:
# Attach the 3 columns to offers, then remove the old JSON column
offers = offers.join(warranty.add_prefix("warranty_")).drop(columns="warranty")

# Clean up
offers["warranty_description"] = offers["warranty_description"].replace("None", pd.NA).map(normalize_fa)
offers["warranty_provider"] = offers["warranty_provider"].map(normalize_fa)
offers["warranty_period_months"] = offers["warranty_period_months"].astype("Int64")

ValueError: columns overlap but no suffix specified: Index(['warranty_description', 'warranty_period_months', 'warranty_provider'], dtype='str')

In [ ]:
# verify
print(offers.columns.tolist())
offers[["warranty_provider", "warranty_period_months", "warranty_description"]].notna().sum()

## 10. Understanding Price: The Most Important Data!


In [ ]:
offers["price_toman"].describe()

OHHHHH WHAT THE HACK IS THAT?! max price is 733518940747298944.0 which I can't even read! Min price is 1 Toman. really?!

Median is normal which is a good sign. average is crazy duo to outliers. We need to understand what is wrong with the data.

In [ ]:
# What product is that crazy number?
offers.nlargest(8, "price_toman")[["seller_title", "price_toman"]]

In [ ]:
# Let's see some statistics to understand data in more details
offers["price_toman"].dropna().astype(str).str.len().value_counts().sort_index()

Ok this is good. Most prices have 6–8 digits.

In [ ]:
# Let's find what product costs 10 Miliyard Toman?!
offers[offers["price_toman"].between(10_000_000_000, 99_999_999_999)][["seller_title", "price_toman"]].sample(8)

There is something wrong with this pricing data. The most expensive real items cost tens of billions, that makes sense. but I consider product above 100 Billion toman as junk!

Now let's investigate the other side.

In [ ]:
offers["price_toman"].value_counts().sort_index().head(10)

Something is wrong here. 635 products are 1 Toman? REALLLLLY?! Which shop? What product?

In [ ]:
# Find which shops set price at 1
offers.loc[offers["price_toman"] == 1, "shop_id"].value_counts().head()

In [ ]:
# Let's see how many product those top 2 shops have?
for shop in ["shop_f50a247a830c0f18192ef5a7", "shop_ace463d318ec387d2e948291"]:
    shop_offers = offers[offers["shop_id"] == shop]
    print(shop, "| products:", shop_offers["product_id"].nunique(),
          "| priced at 1:", (shop_offers["price_toman"] == 1).sum())

Interesting! Those two put 1 on all of their products. This is a placeholder.

In [ ]:
# Show other prices between 2 and 999 Toman
offers[offers["price_toman"].between(2, 999)][["seller_title", "price_toman"]].sample(10)

These products make sense. Tiny electronic products like resistor are cheap.


I prefer to add a status column. 1 is call_for_price, over 100 Miliyard toman is junk, others are ok.


In [ ]:
# Find if we have repeated numbers for placeholder
def is_same_digit(price):
    if pd.isna(price):
        return False
    text = str(int(price))
    return len(text) >= 6 and len(set(text)) == 1


is_repeated = offers["price_toman"].map(is_same_digit)
offers.loc[is_repeated, "price_toman"].value_counts()

In [ ]:
# Add price status
price = offers["price_toman"]

offers["price_status"] = "ok"
offers.loc[price >= 100_000_000_000, "price_status"] = "too_high"
offers.loc[is_repeated, "price_status"] = "placeholder"
offers.loc[price == 1, "price_status"] = "placeholder"
offers.loc[price.isna(), "price_status"] = "missing"

In [ ]:
print(offers["price_status"].value_counts())
print(offers.loc[offers["price_status"] == "ok", "price_toman"].describe())

## 11. Price Freshness

Let's see how old are the data we have. This is also important for Hackathon.


In [ ]:
# what date range do we have?
print(offers["last_price_changed_at"].agg(["min", "max"]))
print(offers["last_price_changed_at"].isna().sum())

In [ ]:
# Which years are prices from?
offers["last_price_changed_at"].dt.year.value_counts().sort_index()


In [ ]:
# price age statistics
snapshot_date = offers["last_price_changed_at"].max()

offers["price_age_days"] = (snapshot_date - offers["last_price_changed_at"]).dt.days.astype("Int64")
offers["price_age_days"].describe(percentiles=[0.25, 0.5, 0.75, 0.9])


Good. On average our price data are for the last 61 days. Acceptable but our agent needs to be aware of this.

Since prices change fast in Iran, I consider the reliability of prices as follows:

- fresh: 7 days or less. Trust it.
- recent: 8–30 days. Probably close.
- old: 31–180 days. Might have changed.
- very_old: more than 180 days. Treat as a rough guess.
- unknown: no date.

Since this data is not changing and is used for Hackathon, I could add a new column to offers, but I feel so bad if I do this :D Because the issue is that if we calculate the freshness right now, after 3, 4 days the data will be wrong!

I will calculate freshness in the tool. It will calculate based on the current date. This is much better and reliable approach.

## 12. Understand Uniqueness & Repeated Values


In [ ]:
# Show (shop, product) pair unique
pair_counts = offers.groupby(["shop_id", "product_id"]).size()
print("Total offers:",f"{len(offers):,}")
print("Total unique pair:",f"{len(pair_counts):,}")


In [ ]:
# Show repeat
pair_counts.value_counts().sort_index().head(6)

Around 9k products are repeated more than once. Some 2 times, some 3, and so on!

In [ ]:
# Show biggest repeat
shop_id, product_id = pair_counts.idxmax()
offers[(offers["shop_id"] == shop_id) & (offers["product_id"] == product_id)][["seller_title", "price_toman"]].head(8)

These are variations. no problem.

I prefer to keep `offers` as it is. But I build a new table called `shop_products` that includes one row per shop + product with its lowest and highest price.

Agent tool for comparison will use this new table.

In [ ]:
# Build the table
offers["ok_price"] = offers["price_toman"].where(offers["price_status"] == "ok")

shop_products = (
    offers.groupby(["shop_id", "product_id"])
    .agg(
        min_price=("ok_price", "min"),
        max_price=("ok_price", "max"),
        variant_count=("offer_id", "size"),
        last_price_changed_at=("last_price_changed_at", "max"),
    )
    .reset_index()
)

In [ ]:
shop_products.head()

## 13. Understand & Handle Address

In [ ]:
# Show some random addresses
shops["address"].sample(10).tolist()

Addresses are separated by comma, going from street to unite.

In [ ]:
# What word each address starts with
parts = shops["address"].dropna().str.split(r"[،,٬]", regex=True).explode().str.strip()
parts.str.split(" ").str[0].value_counts().head(20)

Let's parse all addresses and get the most useful information from them which are `building`, `unit` (واحد), `plaque` (پلاک), and `floor`.

We need to cut the address into pieces, and look at each piece’s first word to decide what it is.

- .startswith(BUILDING_WORDS) works with a whole tuple of words at once.
- .removeprefix("واحد") removes the keyword, so واحد 622 becomes just 622.
- result["building"] is None keeps only the first building we find.

In [ ]:
BUILDING_WORDS = ("پاساژ", "مجتمع", "ساختمان", "بازار", "مرکز خرید", "مرکز تجاری", "برج", "سرای")

def is_floor(part):
    return "طبقه" in part or "همکف" in part or "زیرزمین" in part or part.startswith("منفی")

def parse_address(address):
    result = {"building": None, "floor_text": None, "unit": None, "plaque": None}
    if pd.isna(address):
        return result
    for part in re.split(r"[،,٬]", address):
        part = part.strip()
        if part.startswith("واحد"):
            result["unit"] = part.removeprefix("واحد").strip()
        elif part.startswith("پلاک"):
            result["plaque"] = part.removeprefix("پلاک").strip()
        elif is_floor(part):
            result["floor_text"] = part
        elif part.startswith(BUILDING_WORDS) and result["building"] is None:
            result["building"] = part
    return result

In [ ]:
# Test
parse_address("خیابان جمهوری، نبش شیخ هادی، پاساژ علالدین 2، طبقه 6، واحد 622")

In [ ]:
# Apply to all
parsed = pd.DataFrame(shops["address"].map(parse_address).tolist(), index=shops.index)
shops = shops.join(parsed)

shops[["building", "floor_text", "unit", "plaque"]].notna().sum()

In [ ]:
# Let's look out how floors are written
shops["floor_text"].value_counts().head(20)

Yes ofc we need to normalize!

In [ ]:
FLOOR_WORDS = {"اول": 1, "دوم": 2, "سوم": 3, "چهارم": 4, "پنجم": 5,
               "ششم": 6, "هفتم": 7, "هشتم": 8, "نهم": 9, "دهم": 10}

def floor_to_number(text):
    if pd.isna(text):
        return None
    if "زیر" in text:
        return -1
    if "منفی" in text or "-" in text:
        return -2 if ("2" in text or "دو" in text) else -1
    if "همکف" in text:
        return 0
    for word, number in FLOOR_WORDS.items():
        if word in text:
            return number
    if re.search(r"[BP]\d", text):
        return -int(re.findall(r"\d+", text)[0])
    digits = re.findall(r"\d+", text)
    return int(digits[0]) if digits else None

shops["floor"] = shops["floor_text"].map(floor_to_number).astype("Int64")
shops.loc[shops["floor"] > 20, "floor"] = pd.NA

In [ ]:
# Verify
shops["floor"].value_counts().sort_index()

In [ ]:
shops.loc[shops["floor_text"].notna() & shops["floor"].isna(), "floor_text"].value_counts()

In [ ]:
# Clean up the messy floor_text
shops = shops.drop(columns="floor_text")

Yes I did not forget! We need to NORMALIZE Persian text :)))

In [ ]:
print("Unique building names", shops["building"].nunique())
shops.loc[shops["building"].str.contains("علا", na=False), "building"].value_counts().head(20)


In [ ]:
TYPE_WORDS = ["مرکز خرید", "مرکز تجاری", "پاساژ", "مجتمع", "تجاری", "اداری", "ساختمان"]
NUMBER_WORDS = {"یک": "1", "اول": "1", "دو": "2", "دوم": "2", "سه": "3", "سوم": "3"}

def building_key(name):
    if pd.isna(name):
        return None
    key = name
    for word in TYPE_WORDS:
        key = key.replace(word, " ")
    key = re.sub(r"[()]", " ", key)
    words = [NUMBER_WORDS.get(w, w) for w in key.split()]
    key = "".join(words)
    key = re.sub(r"ا?علا.{0,3}دین", "علاالدین", key)
    return key or None

shops["building_key"] = shops["building"].map(building_key)

# Same name in different places means different buildings, so add a rough location (~1 km)
has_key = shops["building_key"].notna()
shops.loc[has_key, "building_key"] = (
    shops.loc[has_key, "building_key"]
    + "_" + shops.loc[has_key, "lat"].round(2).astype(str)
    + "_" + shops.loc[has_key, "lon"].round(2).astype(str)
)

In [ ]:
print(shops["building_key"].nunique())
shops["building_key"].value_counts().head(10)

## 14. Unpack & Handle Product Spec


In [ ]:
rows = []
for product_id, specs_json in zip(products["product_id"], products["specs"]):
    if pd.isna(specs_json):
        continue
    for spec in json.loads(specs_json):
        rows.append({
            "product_id": product_id,
            "section": spec["section"],
            "key": spec["key"],
            "value": spec["value"],
        })

product_specs = pd.DataFrame(rows)

print("Number of records in product_spec:",f"{len(product_specs):,}")
print("Number of unique products in product_spec:",f"{product_specs["product_id"].nunique():,}")

In [ ]:
product_specs.head()

In [ ]:
# How many specs per product?
product_specs.groupby("product_id").size().describe()

Median is 6 which makes sense but again max has a crazy value.

In [ ]:
# Show what values are in `section` column
product_specs["section"].value_counts()

Almost every row has the same section which is `مشخصات کلی`, so it tells us nothing. I drop it.

In [ ]:
product_specs = product_specs.drop(columns="section").drop_duplicates(["product_id", "key"])
print(len(product_specs))

In [ ]:
# Normalize and add keys
product_specs["key"] = product_specs["key"].map(normalize_fa)
product_specs["value"] = product_specs["value"].map(normalize_fa)

In [ ]:
# Show what is inside key column
print(f"{product_specs['key'].nunique():,}")
product_specs["key"].value_counts().head(15)

This spec now can be used for product comparison. I will add the tool for it.

## 15. Final clean ups


In [ ]:
# Instagram & Website
instagram = shops["instagram"].dropna().map(json.loads)
shops["instagram_url"] = instagram.map(lambda info: info["profile_url"])

print(
    shops["instagram_url"].notna().sum(),
    shops["website_url"].notna().sum()
)

In [ ]:
# Phone numbers
contacts = shops["contacts"].map(json.loads)
shops["phone"] = contacts.map(lambda items: items[0]["display_number"]).map(normalize_fa)

shops["phone"].head()

In [ ]:
# Product images
images = products["image_urls"].dropna().map(json.loads)
products["main_image"] = images.map(lambda urls: urls[0])

print(products["main_image"].notna().sum())

# 16. Save new tables

Finally, this is time to save the cleaned tables.

I use .parquet which is a binary format. It is smaller in size and much faster in loading compared to CSV.

In [ ]:
from pathlib import Path

In [ ]:
shops = shops.drop(columns=["weekly_hours", "payment_options", "category_names", "instagram", "contacts"])
products = products.drop(columns=["specs"])
offers = offers.drop(columns=["ok_price", "price_age_days"])

In [ ]:
clean_dir = Path("clean")

tables = {
    "shops": shops,
    "offers": offers,
    "products": products,
    "shop_products": shop_products,
    "shop_working_hours": shop_working_hours,
    "shop_payments": shop_payments,
    "shop_categories": shop_categories,
    "product_specs": product_specs,
}

for name, table in tables.items():
    table.to_parquet(clean_dir / f"{name}.parquet", index=False)
    print(f"{name}: {len(table):,} rows")